# PhQure — Branch A: URL Feature-Based Detection

This notebook trains two classical ML models (XGBoost and Random Forest) on 29 hand-crafted URL features extracted from ~100,000 URLs.

**Input:** CSV of 29 numerical features per URL 
**Output:** Trained models + SHAP explainability plots 
**Result:** 99.70% accuracy, AUC 0.9997

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix
)
from xgboost import XGBClassifier
import shap
import warnings
warnings.filterwarnings('ignore')

print('All libraries loaded successfully.')

## 2. Configuration

Update `BASE` to point to your local PhQure folder, or leave as-is if running on Google Colab with Drive mounted.

In [ ]:
# Update this path to match your setup
BASE   = os.path.join(os.path.expanduser('~'), 'Desktop', 'PhQure')
DATA   = os.path.join(BASE, 'data', 'branch_a_features.csv')
MODELS = os.path.join(BASE, 'models')
os.makedirs(MODELS, exist_ok=True)

print(f'Base path : {BASE}')
print(f'Data path : {DATA}')
print(f'Models    : {MODELS}')

## 3. Load Data

The feature CSV was generated by `src/extract_features.py`. It contains 29 URL features (length, special characters, IP usage, etc.) and a binary label (0 = legitimate, 1 = phishing).

In [ ]:
df = pd.read_csv(DATA)
X  = df.drop('label', axis=1)
y  = df['label']
feature_names = X.columns.tolist()

print(f'Total samples : {len(df):,}')
print(f'Features      : {len(feature_names)}')
print(f'Phishing      : {y.sum():,} ({y.mean()*100:.1f}%)')
print(f'Legitimate    : {(1-y).sum():,} ({(1-y).mean()*100:.1f}%)')
df.head()

## 4. Train/Test Split

Stratified split ensures both classes are proportionally represented in train and test sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {len(X_train):,}')
print(f'Testing samples  : {len(X_test):,}')

## 5. Evaluation Helper

In [ ]:
def evaluate(name, model, X_test, y_test):
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    print(f'\n{name}')
    print(f'  Accuracy : {acc*100:.2f}%')
    print(f'  F1 Score : {f1*100:.2f}%')
    print(f'  AUC      : {auc:.4f}')
    print(classification_report(y_test, y_pred, target_names=['Legit', 'Phishing']))
    return y_pred, y_proba, acc, f1, auc

## 6. Train XGBoost

XGBoost is a gradient boosting framework. Key hyperparameters:
- `n_estimators=300` — number of trees
- `max_depth=6` — how deep each tree can grow
- `learning_rate=0.1` — step size for each boosting round

In [ ]:
print('Training XGBoost...')
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train, y_train)
xgb_pred, xgb_proba, xgb_acc, xgb_f1, xgb_auc = evaluate('XGBoost', xgb, X_test, y_test)

## 7. Train Random Forest

Random Forest builds many decision trees independently and averages their predictions. Good complement to XGBoost as it uses bagging instead of boosting.

In [ ]:
print('Training Random Forest...')
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_pred, rf_proba, rf_acc, rf_f1, rf_auc = evaluate('Random Forest', rf, X_test, y_test)

## 8. Save Models

In [ ]:
joblib.dump(xgb, os.path.join(MODELS, 'xgboost_branchA.pkl'))
joblib.dump(rf,  os.path.join(MODELS, 'randomforest_branchA.pkl'))
print('Models saved to models/')

## 9. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Branch A — Confusion Matrices', fontsize=14, fontweight='bold')

for ax, name, pred in zip(axes, ['XGBoost', 'Random Forest'], [xgb_pred, rf_pred]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legit', 'Phishing'],
                yticklabels=['Legit', 'Phishing'])
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig(os.path.join(BASE, 'data', 'branchA_confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. SHAP Explainability

SHAP (SHapley Additive exPlanations) shows which features pushed each prediction toward phishing or legitimate. The summary plot shows both direction and magnitude of each feature's impact.

In [ ]:
print('Generating SHAP values (2-3 minutes)...')
explainer   = shap.TreeExplainer(xgb)
shap_sample = X_test.sample(n=1000, random_state=42)
shap_values = explainer.shap_values(shap_sample)

# Summary plot (beeswarm)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, shap_sample, feature_names=feature_names, show=False)
plt.title('Branch A — SHAP Feature Importance (XGBoost)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'data', 'branchA_shap_summary.png'), dpi=150, bbox_inches='tight')
plt.show()

# Bar plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, shap_sample, feature_names=feature_names, plot_type='bar', show=False)
plt.title('Branch A — SHAP Mean Feature Importance (XGBoost)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'data', 'branchA_shap_bar.png'), dpi=150, bbox_inches='tight')
plt.show()

## 11. Final Results Summary

In [ ]:
results = pd.DataFrame({
    'Model':    ['XGBoost', 'Random Forest'],
    'Accuracy': [f'{xgb_acc*100:.2f}%', f'{rf_acc*100:.2f}%'],
    'F1 Score': [f'{xgb_f1*100:.2f}%', f'{rf_f1*100:.2f}%'],
    'AUC':      [f'{xgb_auc:.4f}', f'{rf_auc:.4f}']
})
print('Branch A — Final Results')
print(results.to_string(index=False))